# Cyber Surveillance System — Google Colab

هذا الدفتر يشغّل منصة Python المركزية في Colab ويفتح واجهة المختبر عبر رابط HTTPS مؤقت. لا تحتاج إلى تثبيت PyTorch على اللابتوب.

**قبل البدء:** من Runtime → Change runtime type اختر GPU إن كان متاحًا، ثم شغّل الخلايا بالترتيب. عند فتح الواجهة ألصق مفتاح API الذي أدخلته في الدفتر في حقل «مفتاح API» ثم انقر خارج الحقل.

> هذه بيئة اختبار مؤقتة وليست خادمًا دائمًا. رابط النفق عام لمن يملكه؛ واجهات API محمية بمفتاح الجلسة. لا تشارك المفتاح ولا ترفع فيديوهات لا تملك تصريحًا باستخدامها.

In [ ]:
import os, sys, platform, subprocess
print("Python:", sys.version)
print("Platform:", platform.platform())
gpu = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout[:4000] if gpu.returncode == 0 else "No NVIDIA GPU detected; CPU mode is possible but slower.")

## 1. تنزيل المستودع من main

In [ ]:
from pathlib import Path
REPO = Path("/content/Cyber-surveillance-system")
URL = "https://github.com/CybersecurityRahbar/Cyber-surveillance-system.git"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
%cd /content/Cyber-surveillance-system

## 2. تثبيت الفيديو ومحرك الرؤية

تستخدم هذه الجلسة نسخة PyTorch الموجودة في صورة Colab إن كانت متوافقة، ثم تثبت PyAV وUltralytics. أوزان YOLO26n تُنزّل عند أول تحليل فيديو.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[video]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.197"], check=True)

import torch, av, cv2, ultralytics
print("Torch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("PyAV:", av.__version__, "| OpenCV:", cv2.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. مكان حفظ الفيديوهات وقاعدة البيانات

افتراضيًا بيانات Colab مؤقتة وتضيع عند انتهاء Runtime. غيّر المتغير إلى True لحفظ البيانات في Google Drive.

In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/CyberSurveillanceLab")
else:
    DATA = Path("/content/cyber-surveillance-data")
(DATA / "lab").mkdir(parents=True, exist_ok=True)
os.environ["TRAFFIC_SURVEILLANCE_DATABASE"] = str(DATA / "surveillance.sqlite3")
os.environ["TRAFFIC_SURVEILLANCE_LAB_STORAGE_DIR"] = str(DATA / "lab")
os.environ["TRAFFIC_SURVEILLANCE_MAX_UPLOAD_MB"] = "1024"
print("Data directory:", DATA)

## 4. إعداد مفتاح حماية API

أدخل مفتاحًا عشوائيًا خاصًا بك، لا يقل عن 32 حرفًا. ستحتاج إلى إدخال القيمة نفسها في حقل «مفتاح API» في واجهة المتصفح. الإدخال مخفي ولا يُطبع في الدفتر.

In [ ]:
from getpass import getpass
API_KEY = getpass("Private API key (32+ characters): ")
if len(API_KEY) < 32:
    raise ValueError("API key must contain at least 32 characters.")
os.environ["TRAFFIC_SURVEILLANCE_API_KEY"] = API_KEY
os.environ["TRAFFIC_SURVEILLANCE_ENVIRONMENT"] = "development"
os.environ["TRAFFIC_SURVEILLANCE_HOST"] = "127.0.0.1"
print("API protection configured.")

## 5. تشغيل خادم المنصة داخل Colab

In [ ]:
import time, requests
SERVER_LOG = Path("/tmp/traffic-surveillance-server.log")
server_log = SERVER_LOG.open("w")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "traffic_surveillance.api.app:app",
     "--host", "127.0.0.1", "--port", "8000", "--workers", "1"],
    cwd=str(REPO), env=os.environ.copy(), stdout=server_log, stderr=subprocess.STDOUT
)
for _ in range(45):
    if server_proc.poll() is not None:
        raise RuntimeError(SERVER_LOG.read_text(errors="replace")[-5000:])
    try:
        r = requests.get("http://127.0.0.1:8000/healthz", timeout=2)
        if r.ok:
            print("Server ready:", r.json())
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise TimeoutError("Uvicorn did not start; inspect " + str(SERVER_LOG))

## 6. نشر الواجهة عبر Cloudflare Quick Tunnel

لا يتطلب هذا النفق حساب Cloudflare أو نطاقًا. الرابط مؤقت ويتوقف عند إيقاف cloudflared أو انتهاء Runtime. يمكنك اختياريًا تقييد الدخول ببريد إلكتروني.

In [ ]:
import re
from IPython.display import Markdown, display

CLOUDFLARE_EMAIL = ""  # اختياري؛ اتركه فارغًا للرابط العام
DEB = "/tmp/cloudflared.deb"
subprocess.run(["bash", "-lc",
    f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O {DEB} && dpkg -i {DEB}"],
    check=True)

cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"]
if CLOUDFLARE_EMAIL.strip():
    cmd += ["--allowed-mail", CLOUDFLARE_EMAIL.strip()]
CF_LOG = Path("/tmp/cloudflared.log")
cf_log = CF_LOG.open("w")
cloudflared_proc = subprocess.Popen(cmd, stdout=cf_log, stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(90):
    if cloudflared_proc.poll() is not None:
        break
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                      CF_LOG.read_text(errors="replace"))
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(1)
if not PUBLIC_URL:
    raise RuntimeError("Cloudflare URL not found:\n" + CF_LOG.read_text(errors="replace")[-5000:])
print("Tunnel URL:", PUBLIC_URL)
display(Markdown(f"### [افتح واجهة المختبر]({PUBLIC_URL}/lab)"))

## 7. استخدام المختبر

1. افتح الرابط المنشور مع المسار `/lab`.
2. ألصق مفتاح API نفسه في حقل «مفتاح API»، ثم انقر خارج الحقل؛ ستعيد الواجهة فحص حالة الاتصال.
3. اختر فيديو قصيرًا (20–30 ثانية كبداية) وابدأ التحليل.
4. راقب الكشف والمربعات الخضراء وTrack IDs والعدادات وFPS، ثم شغّل MP4 الناتج أو نزّله.

صفحة HTML والملفات الثابتة عامة لمن يعرف الرابط؛ لكن عمليات API ورفع الفيديو محمية بمفتاح API. لا تشارك المفتاح.

## 8. بديل اختياري: ngrok

استخدم هذه الخلية بدل Cloudflare، ولا تشغّل النفقين معًا. تحتاج حساب ngrok وAuth Token من لوحة حسابك؛ أدخل الرمز في prompt المخفي ولا تضعه في الشيفرة أو GitHub.

In [ ]:
if "cloudflared_proc" in globals() and cloudflared_proc.poll() is None:
    cloudflared_proc.terminate()
    cloudflared_proc.wait(timeout=10)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
from pyngrok import ngrok
NGROK_TOKEN = getpass("ngrok Auth Token (hidden): ")
ngrok.set_auth_token(NGROK_TOKEN)
ngrok_tunnel = ngrok.connect(8000, "http")
NGROK_URL = ngrok_tunnel.public_url
print("ngrok URL:", NGROK_URL)
display(Markdown(f"### [افتح واجهة المختبر عبر ngrok]({NGROK_URL}/lab)"))

## سجلات التشغيل والإيقاف

عند حدوث خطأ اقرأ السجل. عند الانتهاء أوقف الخادم والنفق. تذكّر أن Colab مجانيًا مورد تفاعلي؛ التوفر والمدة وGPU غير مضمونة، لذلك لا تستخدمه كنشر إنتاجي على مدار الساعة.

In [ ]:
print("=== Uvicorn ===")
print(SERVER_LOG.read_text(errors="replace")[-6000:])
print("\n=== Cloudflare ===")
if CF_LOG.exists():
    print(CF_LOG.read_text(errors="replace")[-6000:])

# Stop when finished:
# if cloudflared_proc.poll() is None: cloudflared_proc.terminate()
# if server_proc.poll() is None: server_proc.terminate()
# ngrok.kill()  # only if ngrok was used